Phase 00 Environment Setup & Docker Stack

In [1]:
import pyspark
import duckdb
import boto3

print('PySpark:', pyspark.__version__)
print('DuckDB:', duckdb.__version__)
print('boto3:', boto3.__version__)

PySpark: 3.5.1
DuckDB: 0.10.3
boto3: 1.34.0


Phase 02 PySpark ETL — Bronze → Silver


In [3]:
import duckdb; 
con = duckdb.connect(); 
con.execute('INSTALL httpfs; LOAD httpfs;'); 
con.execute("SET s3_endpoint='localhost:9000'; SET s3_use_ssl=false; SET s3_url_style='path'; SET s3_access_key_id='sensorflow'; SET s3_secret_access_key='sensorflow123';"); 
n = con.execute("SELECT COUNT(*) FROM read_parquet('s3://sensorflow-local/processed/**/*.parquet')").fetchone()[0]; 
print('Silver rows:', n)
# Expected: ~240 per 60-second simulator run

Silver rows: 60807


Phase 03 Gold Layer — DuckDB Star Schema

In [6]:
import duckdb

# Connect to the DuckDB database
con = duckdb.connect('data/sensorflow.db', read_only=True)

# List of query files to execute
query_files = [
    'queries/BQ-01.sql',
    'queries/BQ-02.sql',
    'queries/BQ-03.sql',
    'queries/BQ-04.sql'
]

# Loop through and execute each file
for file_path in query_files:
    print(f"=== Results for {file_path} ===")
    try:
        # Read the SQL query
        with open(file_path, 'r') as file:
            query = file.read()
        
        # Execute and print as a pandas DataFrame
        result_df = con.execute(query).df()
        print(result_df)
        
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"Error executing '{file_path}': {e}")
    
    print("\n")  # Add a blank line between results for readability

# Close the connection when done
con.close()

=== Results for queries/BQ-01.sql ===
   machine_id  max_temp  high_count  total_readings
0  MACHINE_02    103.49         355            3046
1  MACHINE_03    103.49         354            3016
2  MACHINE_05    103.48         350            3046
3  MACHINE_01    103.48         341            3053
4  MACHINE_04    103.44         320            3046


=== Results for queries/BQ-02.sql ===
Empty DataFrame
Columns: [machine_id, actual_readings, uptime_pct]
Index: []


=== Results for queries/BQ-03.sql ===
       shift  total_readings  anomalies  anomaly_pct
0  afternoon           23064       5031        21.81
1      night           13222        666         5.04
2    morning           24521       1164         4.75


=== Results for queries/BQ-04.sql ===
       machine_id reading_date  rolling_avg_7d
0      MACHINE_01   2026-05-04       82.720000
1      MACHINE_01   2026-05-04       73.365000
2      MACHINE_01   2026-05-04       75.626667
3      MACHINE_01   2026-05-04       74.595000
4     

Phase 05 CDC & Nightly Aggregation

In [1]:
import duckdb; 
con=duckdb.connect('data/sensorflow.db', read_only=True); 
print(con.execute('SELECT COUNT(*) FROM anomaly_log').fetchone())
con.close()

(3848,)


Phase 06 Dashboards — Grafana, Superset & Plotly

In [3]:
import duckdb
con = duckdb.connect('data/sensorflow.db', read_only=True)
tables = con.execute("SHOW TABLES").fetchall()
print('Tables:', [r[0] for r in tables])
for t in [r[0] for r in tables]:
    n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n} rows')
con.close()

Tables: ['anomaly_log', 'dim_date', 'dim_location', 'dim_machine', 'dim_sensor', 'fact_sensor_readings']
  anomaly_log: 3848 rows
  dim_date: 1096 rows
  dim_location: 3 rows
  dim_machine: 5 rows
  dim_sensor: 4 rows
  fact_sensor_readings: 49247 rows
